# SQL Capstone Project — Group 1
## E-Commerce Operations & Customer Experience

**Unified Team Notebook — Google Drive + SQLite**

### Task Allocation
- **Task 1, 6, 10:** لقاء
- **Task 2, 7, 11:** روان موسى
- **Task 3, 8, 12:** روان إبراهيم
- **Task 4, 5, 9:** إبراهيم


# Project Setup — Rewan Ibrahim
This is the common setup used by all 12 tasks.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import sqlite3
import pandas as pd

In [ ]:
BASE_PATH = "/content/drive/MyDrive/SQL_Capstone_Group1_Rewan"
DATA_PATH = f"{BASE_PATH}/data"
DB_PATH = f"{BASE_PATH}/olist.db"

In [ ]:
os.makedirs(DATA_PATH, exist_ok=True)

In [ ]:
conn = sqlite3.connect(DB_PATH)

print("Connected successfully")
print(DB_PATH)

In [ ]:
tables = {
    "customers": "olist_customers_dataset.csv",
    "geolocation": "olist_geolocation_dataset.csv",
    "order_items": "olist_order_items_dataset.csv",
    "order_payments": "olist_order_payments_dataset.csv",
    "order_reviews": "olist_order_reviews_dataset.csv",
    "orders": "olist_orders_dataset.csv",
    "products": "olist_products_dataset.csv",
    "sellers": "olist_sellers_dataset.csv",
    "category_translation": "product_category_name_translation.csv"
}

In [ ]:
for table_name, file_name in tables.items():

    table_exists = conn.execute(
        """
        SELECT name
        FROM sqlite_master
        WHERE type='table' AND name=?
        """,
        (table_name,)
    ).fetchone()

    if table_exists:
        print(f"{table_name} already exists - skipped")

    else:
        file_path = f"{DATA_PATH}/{file_name}"

        df = pd.read_csv(file_path)

        df.to_sql(
            table_name,
            conn,
            if_exists="fail",
            index=False
        )

        print(f"{table_name}: {len(df):,} rows loaded")

In [ ]:
query = """
SELECT
    name AS table_name
FROM sqlite_master
WHERE type = 'table'
ORDER BY name;
"""

pd.read_sql_query(query, conn)

In [ ]:
for table_name in tables.keys():

    count = pd.read_sql_query(
        f"SELECT COUNT(*) AS row_count FROM {table_name}",
        conn
    )

    print(
        table_name,
        "→",
        f"{count.loc[0, 'row_count']:,}",
        "rows"
    )

In [ ]:
for table_name in [
    "orders",
    "order_items",
    "order_payments",
    "order_reviews",
    "products"
]:
    print(f"\n===== {table_name} =====")

    df = pd.read_sql_query(
        f"SELECT * FROM {table_name} LIMIT 5",
        conn
    )

    display(df)

In [ ]:
key_checks = {
    "orders": "order_id",
    "order_items": "order_id",
    "order_payments": "order_id",
    "order_reviews": "order_id",
    "products": "product_id"
}

for table_name, key_column in key_checks.items():

    query = f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT {key_column}) AS distinct_keys,
        COUNT(*) - COUNT(DISTINCT {key_column}) AS duplicate_key_rows
    FROM {table_name};
    """

    result = pd.read_sql_query(query, conn)

    print(f"\n===== {table_name} =====")
    display(result)

In [ ]:
queries = {
    "order_items": """
        SELECT
            order_id,
            COUNT(*) AS rows_per_order
        FROM order_items
        GROUP BY order_id
        ORDER BY rows_per_order DESC
        LIMIT 10;
    """,

    "order_payments": """
        SELECT
            order_id,
            COUNT(*) AS rows_per_order
        FROM order_payments
        GROUP BY order_id
        ORDER BY rows_per_order DESC
        LIMIT 10;
    """,

    "order_reviews": """
        SELECT
            order_id,
            COUNT(*) AS rows_per_order
        FROM order_reviews
        GROUP BY order_id
        ORDER BY rows_per_order DESC
        LIMIT 10;
    """
}

for name, query in queries.items():
    print(f"\n===== {name} =====")
    display(pd.read_sql_query(query, conn))

# Task 1 — Data Quality and Key Audit — Owner: لقاء


Orders Table

In [ ]:
query = """
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT order_id) AS unique_orders,
    SUM(CASE WHEN order_id IS NULL THEN 1 ELSE 0 END) AS null_orders,
    SUM(CASE WHEN order_id IS NULL THEN 1 ELSE 0 END) * 100.0 / COUNT(*) AS null_order_rate
FROM orders;
"""

pd.read_sql_query(query, conn)


Check duplicate values

In [ ]:
query = """
SELECT
    order_id,
    COUNT(*) AS order_count
FROM orders
WHERE order_id IS NOT NULL
GROUP BY order_id
HAVING COUNT(*) > 1;
"""

pd.read_sql_query(query, conn)


Grain: One row represents one order.

Order Items Table

In [ ]:
query = """
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT order_id) AS unique_orders,
    SUM(CASE WHEN order_id IS NULL THEN 1 ELSE 0 END) AS null_orders,
    SUM(CASE WHEN order_id IS NULL THEN 1 ELSE 0 END) * 100.0 / COUNT(*) AS null_order_rate,
    SUM(CASE WHEN order_item_id IS NULL THEN 1 ELSE 0 END) AS null_items,
    SUM(CASE WHEN order_item_id IS NULL THEN 1 ELSE 0 END) * 100.0 / COUNT(*) AS null_item_rate
FROM order_items;
"""

pd.read_sql_query(query, conn)


Check duplicate values

In [ ]:
query = """
SELECT
    order_id,
    order_item_id,
    COUNT(*) AS item_count
FROM order_items
WHERE order_id IS NOT NULL
  AND order_item_id IS NOT NULL
GROUP BY order_id, order_item_id
HAVING COUNT(*) > 1;
"""

pd.read_sql_query(query, conn)


Grain:One row represents one item within an order.

Order Payments Table

In [ ]:
query = """
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT order_id) AS unique_orders,
    SUM(CASE WHEN order_id IS NULL THEN 1 ELSE 0 END) AS null_orders,
    SUM(CASE WHEN order_id IS NULL THEN 1 ELSE 0 END) * 100.0 / COUNT(*) AS null_order_rate,
    SUM(CASE WHEN payment_sequential IS NULL THEN 1 ELSE 0 END) AS null_payments,
    SUM(CASE WHEN payment_sequential IS NULL THEN 1 ELSE 0 END) * 100.0 / COUNT(*) AS null_payment_rate
FROM order_payments;
"""

pd.read_sql_query(query, conn)


Check duplicate values

In [ ]:
query = """
SELECT
    order_id,
    payment_sequential,
    COUNT(*) AS payment_count
FROM order_payments
WHERE order_id IS NOT NULL
  AND payment_sequential IS NOT NULL
GROUP BY order_id, payment_sequential
HAVING COUNT(*) > 1;
"""

pd.read_sql_query(query, conn)


Grain: One row represents one payment record for an order.

Customers Table

In [ ]:
query = """
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT customer_id) AS unique_customers,
    SUM(CASE WHEN customer_id IS NULL THEN 1 ELSE 0 END) AS null_customers,
    SUM(CASE WHEN customer_id IS NULL THEN 1 ELSE 0 END) * 100.0 / COUNT(*) AS null_customer_rate
FROM customers;
"""

pd.read_sql_query(query, conn)


Check duplicate values

In [ ]:
query = """
SELECT
    customer_id,
    COUNT(*) AS customer_count
FROM customers
WHERE customer_id IS NOT NULL
GROUP BY customer_id
HAVING COUNT(*) > 1;
"""

pd.read_sql_query(query, conn)


Grain: One row represents one customer.

Products Table

In [ ]:
query = """
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT product_id) AS unique_products,
    SUM(CASE WHEN product_id IS NULL THEN 1 ELSE 0 END) AS null_products,
    SUM(CASE WHEN product_id IS NULL THEN 1 ELSE 0 END) * 100.0 / COUNT(*) AS null_product_rate
FROM products;
"""

pd.read_sql_query(query, conn)


Check duplicate values

In [ ]:
query = """
SELECT
    product_id,
    COUNT(*) AS product_count
FROM products
WHERE product_id IS NOT NULL
GROUP BY product_id
HAVING COUNT(*) > 1;
"""

pd.read_sql_query(query, conn)


Grain: One row represents one product.

Sellers Table

In [ ]:
query = """
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT seller_id) AS unique_sellers,
    SUM(CASE WHEN seller_id IS NULL THEN 1 ELSE 0 END) AS null_sellers,
    SUM(CASE WHEN seller_id IS NULL THEN 1 ELSE 0 END) * 100.0 / COUNT(*) AS null_seller_rate
FROM sellers;
"""

pd.read_sql_query(query, conn)


Check duplicate values

In [ ]:
query = """
SELECT
    seller_id,
    COUNT(*) AS seller_count
FROM sellers
WHERE seller_id IS NOT NULL
GROUP BY seller_id
HAVING COUNT(*) > 1;
"""

pd.read_sql_query(query, conn)


Grain: One row represents one seller.

Order Reviews Table

In [ ]:
query = """
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT review_id) AS unique_reviews,
    SUM(CASE WHEN review_id IS NULL THEN 1 ELSE 0 END) AS null_reviews,
    SUM(CASE WHEN review_id IS NULL THEN 1 ELSE 0 END) * 100.0 / COUNT(*) AS null_review_rate
FROM order_reviews;
"""

pd.read_sql_query(query, conn)


Check duplicate values

In [ ]:
query = """
SELECT
    review_id,
    COUNT(*) AS review_count
FROM order_reviews
WHERE review_id IS NOT NULL
GROUP BY review_id
HAVING COUNT(*) > 1;
"""

pd.read_sql_query(query, conn)


Grain: One row represents one review record.

Category Translation Table

In [ ]:
query = """
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT product_category_name) AS unique_categories,
    SUM(CASE WHEN product_category_name IS NULL THEN 1 ELSE 0 END) AS null_categories,
    SUM(CASE WHEN product_category_name IS NULL THEN 1 ELSE 0 END) * 100.0 / COUNT(*) AS null_category_rate
FROM category_translation;
"""

pd.read_sql_query(query, conn)


Check duplicate values

In [ ]:
query = """
SELECT
    product_category_name,
    COUNT(*) AS category_count
FROM category_translation
WHERE product_category_name IS NOT NULL
GROUP BY product_category_name
HAVING COUNT(*) > 1;
"""

pd.read_sql_query(query, conn)


Grain: One row represents one product category translation.

# Task 2 — Executive KPI Layer — Owner: روان موسى


In [ ]:
task2 = pd.read_sql_query("""
SELECT
    (SELECT COUNT(DISTINCT order_id)
     FROM orders) AS total_orders,

    (SELECT COUNT(DISTINCT order_id)
     FROM orders
     WHERE order_status = 'delivered') AS delivered_orders,

    (SELECT COUNT(DISTINCT order_id)
     FROM orders
     WHERE order_status = 'canceled') AS canceled_orders,

    (SELECT SUM(price)
     FROM order_items) AS gross_item_revenue,

    (SELECT SUM(freight_value)
     FROM order_items) AS total_freight,

    (SELECT SUM(payment_value)
     FROM order_payments) AS total_payment_value,

    ROUND(
        (SELECT SUM(payment_value)
         FROM order_payments) * 1.0
        /
        (SELECT COUNT(DISTINCT order_id)
         FROM orders),
        2
    ) AS average_order_value,

    ROUND(
        (SELECT COUNT(*)
         FROM order_items) * 1.0
        /
        (SELECT COUNT(DISTINCT order_id)
         FROM orders),
        2
    ) AS average_items_per_order
""", conn)

task2


# Task 3 — Monthly Business Trend — Owner: روان إبراهيم


In [ ]:
query_task3_order_level = """
WITH payment_per_order AS (
    SELECT
        order_id,
        SUM(payment_value) AS order_revenue
    FROM order_payments
    GROUP BY order_id
)

SELECT
    o.order_id,
    o.order_purchase_timestamp,
    p.order_revenue
FROM orders AS o
LEFT JOIN payment_per_order AS p
    ON o.order_id = p.order_id
LIMIT 10;
"""

pd.read_sql_query(query_task3_order_level, conn)

In [ ]:
query_task3_monthly = """
WITH payment_per_order AS (
    SELECT
        order_id,
        SUM(payment_value) AS order_revenue
    FROM order_payments
    GROUP BY order_id
)

SELECT
    strftime('%Y-%m', o.order_purchase_timestamp) AS month,

    COUNT(o.order_id) AS total_orders,

    ROUND(
        SUM(p.order_revenue),
        2
    ) AS total_revenue,

    ROUND(
        AVG(p.order_revenue),
        2
    ) AS avg_order_value

FROM orders AS o
LEFT JOIN payment_per_order AS p
    ON o.order_id = p.order_id

GROUP BY
    strftime('%Y-%m', o.order_purchase_timestamp)

ORDER BY
    month;
"""

monthly_trend = pd.read_sql_query(
    query_task3_monthly,
    conn
)

display(monthly_trend)

In [ ]:
monthly_trend_analysis = monthly_trend.copy()

monthly_trend_analysis["order_growth_pct"] = (
    monthly_trend_analysis["total_orders"]
    .pct_change() * 100
)

monthly_trend_analysis["revenue_growth_pct"] = (
    monthly_trend_analysis["total_revenue"]
    .pct_change() * 100
)

monthly_trend_analysis["aov_growth_pct"] = (
    monthly_trend_analysis["avg_order_value"]
    .pct_change() * 100
)

monthly_trend_analysis = monthly_trend_analysis.round(2)

display(monthly_trend_analysis)

In [ ]:
def classify_trend(value):
    if pd.isna(value):
        return "No Previous Month"
    elif value > 0:
        return "Growth"
    elif value < 0:
        return "Decline"
    else:
        return "Stable"

monthly_trend_analysis["revenue_trend"] = (
    monthly_trend_analysis["revenue_growth_pct"]
    .apply(classify_trend)
)

display(
    monthly_trend_analysis[
        [
            "month",
            "total_orders",
            "total_revenue",
            "avg_order_value",
            "revenue_growth_pct",
            "revenue_trend"
        ]
    ]
)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 5))

plt.plot(
    monthly_trend_analysis["month"],
    monthly_trend_analysis["total_orders"],
    marker="o"
)

plt.title("Monthly Orders Trend")
plt.xlabel("Month")
plt.ylabel("Total Orders")
plt.xticks(rotation=45)
plt.tight_layout()

plt.show()

In [ ]:
plt.figure(figsize=(12, 5))

plt.plot(
    monthly_trend_analysis["month"],
    monthly_trend_analysis["total_revenue"],
    marker="o"
)

plt.title("Monthly Revenue Trend")
plt.xlabel("Month")
plt.ylabel("Total Revenue")
plt.xticks(rotation=45)
plt.tight_layout()

plt.show()

In [ ]:
plt.figure(figsize=(12, 5))

plt.plot(
    monthly_trend_analysis["month"],
    monthly_trend_analysis["avg_order_value"],
    marker="o"
)

plt.title("Monthly Average Order Value")
plt.xlabel("Month")
plt.ylabel("Average Order Value")
plt.xticks(rotation=45)
plt.tight_layout()

plt.show()

# Task 4 — Category Performance — Owner: إبراهيم


In [ ]:
query = """
SELECT
    p.product_category_name AS category,
    COUNT(DISTINCT oi.order_id) AS order_count
FROM order_items oi
JOIN products p
    ON oi.product_id = p.product_id
GROUP BY p.product_category_name
ORDER BY order_count DESC;
"""

result = pd.read_sql_query(query, conn)
result.head(20)


In [ ]:
query = """
SELECT
    p.product_category_name AS category,
    COUNT(*) AS units_sold
FROM order_items oi
JOIN products p
    ON oi.product_id = p.product_id
GROUP BY p.product_category_name
ORDER BY units_sold DESC;
"""

result = pd.read_sql_query(query, conn)
result.head(20)


In [ ]:
query = """
SELECT
    p.product_category_name AS category,
    ROUND(SUM(oi.price), 2) AS total_revenue
FROM order_items oi
JOIN products p
    ON oi.product_id = p.product_id
GROUP BY p.product_category_name
ORDER BY total_revenue DESC;
"""

result = pd.read_sql_query(query, conn)
result.head(20)


In [ ]:
query_task_4 = """
SELECT 
    COALESCE(t.product_category_name_english, p.product_category_name) AS category_name,
    COUNT(DISTINCT oi.order_id) AS total_orders,
    COUNT(oi.order_item_id) AS total_units_sold,
    ROUND(SUM(oi.price), 2) AS total_revenue,
    ROUND(AVG(oi.price), 2) AS average_price,
    ROUND(AVG(oi.freight_value), 2) AS average_freight
FROM order_items oi
JOIN products p ON oi.product_id = p.product_id
LEFT JOIN category_translation t ON p.product_category_name = t.product_category_name
GROUP BY category_name
HAVING COUNT(DISTINCT oi.order_id) >= 50
ORDER BY total_revenue DESC;
"""

df_category_performance = pd.read_sql_query(query_task_4, conn)
df_category_performance.head(10)


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# رسم أعلى 10 فئات من حيث الإيرادات (Task 4)
plt.figure(figsize=(12, 6))
top_10_cats = df_category_performance.head(10)

sns.barplot(
    data=top_10_cats, 
    x='total_revenue', 
    y='category_name', 
    palette='Blues_r'
)

plt.title('Top 10 Product Categories by Total Revenue (Task 4)', fontsize=14, fontweight='bold')
plt.xlabel('Total Revenue', fontsize=12)
plt.ylabel('Category Name', fontsize=12)
plt.tight_layout()
plt.show()


# Task 5 — Payment Behavior — Owner: إبراهيم


In [ ]:
# Task 5: تحليل سلوك طرق الدفع وقيمتها وعدد الأقساط
query_task_5 = """
SELECT 
    payment_type,
    COUNT(DISTINCT order_id) AS total_orders,
    ROUND(SUM(payment_value), 2) AS total_payment_value,
    ROUND(AVG(payment_value), 2) AS average_order_value,
    ROUND(AVG(payment_installments), 1) AS avg_installments
FROM order_payments
GROUP BY payment_type
ORDER BY total_payment_value DESC;
"""

df_payment_behavior = pd.read_sql_query(query_task_5, conn)
df_payment_behavior


# Task 6 — Seller Performance — Owner: لقاء


1. Seller Sales

orders + revenue + freight.

In [ ]:
query = """
SELECT
    seller_id,
    COUNT(DISTINCT order_id) AS total_orders,
    SUM(price) AS item_revenue,
    SUM(freight_value) AS total_freight
FROM order_items
GROUP BY seller_id
HAVING COUNT(DISTINCT order_id) >= 10
ORDER BY total_orders DESC;
"""

pd.read_sql_query(query, conn)


2. Unique Customers

In [ ]:
query = """
SELECT
    oi.seller_id,
    COUNT(DISTINCT o.customer_id) AS unique_customers
FROM order_items oi
JOIN orders o
    ON oi.order_id = o.order_id
GROUP BY oi.seller_id;
"""

pd.read_sql_query(query, conn)


3. Average Review Score

In [ ]:
query = """
SELECT
    oi.seller_id,
    AVG(r.review_score) AS average_review_score
FROM (
    SELECT DISTINCT seller_id, order_id
    FROM order_items
) oi
JOIN order_reviews r
    ON oi.order_id = r.order_id
GROUP BY oi.seller_id;
"""

pd.read_sql_query(query, conn)


4. Late Delivery Rate

In [ ]:
query = """
SELECT
    oi.seller_id,
    AVG(
        CASE
            WHEN o.order_delivered_customer_date IS NOT NULL
             AND o.order_estimated_delivery_date IS NOT NULL
            THEN
                CASE
                    WHEN o.order_delivered_customer_date >
                         o.order_estimated_delivery_date
                    THEN 1.0
                    ELSE 0.0
                END
        END
    ) * 100 AS late_delivery_rate
FROM (
    SELECT DISTINCT seller_id, order_id
    FROM order_items
) oi
JOIN orders o
    ON oi.order_id = o.order_id
GROUP BY oi.seller_id;
"""

pd.read_sql_query(query, conn)


5. Final Seller Performance Table

In [ ]:
query = """
WITH seller_sales AS (
    SELECT
        seller_id,
        COUNT(DISTINCT order_id) AS total_orders,
        SUM(price) AS item_revenue,
        SUM(freight_value) AS total_freight
    FROM order_items
    GROUP BY seller_id
    HAVING COUNT(DISTINCT order_id) >= 10
),

seller_customers AS (
    SELECT
        oi.seller_id,
        COUNT(DISTINCT o.customer_id) AS unique_customers
    FROM order_items oi
    JOIN orders o
        ON oi.order_id = o.order_id
    GROUP BY oi.seller_id
),

seller_reviews AS (
    SELECT
        oi.seller_id,
        AVG(r.review_score) AS average_review_score
    FROM (
        SELECT DISTINCT seller_id, order_id
        FROM order_items
    ) oi
    JOIN order_reviews r
        ON oi.order_id = r.order_id
    GROUP BY oi.seller_id
),

seller_delivery AS (
    SELECT
        oi.seller_id,
        AVG(
            CASE
                WHEN o.order_delivered_customer_date IS NOT NULL
                 AND o.order_estimated_delivery_date IS NOT NULL
                THEN
                    CASE
                        WHEN o.order_delivered_customer_date >
                             o.order_estimated_delivery_date
                        THEN 1.0
                        ELSE 0.0
                    END
            END
        ) * 100 AS late_delivery_rate
    FROM (
        SELECT DISTINCT seller_id, order_id
        FROM order_items
    ) oi
    JOIN orders o
        ON oi.order_id = o.order_id
    GROUP BY oi.seller_id
)

SELECT
    s.seller_id,
    s.total_orders,
    c.unique_customers,
    s.item_revenue,
    s.total_freight,
    r.average_review_score,
    d.late_delivery_rate
FROM seller_sales s
LEFT JOIN seller_customers c
    ON s.seller_id = c.seller_id
LEFT JOIN seller_reviews r
    ON s.seller_id = r.seller_id
LEFT JOIN seller_delivery d
    ON s.seller_id = d.seller_id
ORDER BY s.total_orders DESC;
"""

seller_df = pd.read_sql_query(query, conn)
seller_df

top_sellers = seller_df.sort_values("item_revenue", ascending=False).head(10)

top_sellers


📊 Seller Performance — Item Revenue

In [ ]:
import matplotlib.pyplot as plt

plt.barh(top_sellers["seller_id"], top_sellers["item_revenue"])

plt.xlabel("Item Revenue")
plt.ylabel("Seller")
plt.title("Top 10 Sellers by Item Revenue")

plt.show()


# Task 7 — Delivery SLA Analysis — Owner: روان موسى


In [ ]:
task7_delivery = pd.read_sql_query("""
SELECT
    order_id,

    ROUND(
        julianday(order_delivered_customer_date)
        - julianday(order_purchase_timestamp),
        2
    ) AS delivery_days,

    ROUND(
        julianday(order_delivered_customer_date)
        - julianday(order_estimated_delivery_date),
        2
    ) AS days_late_or_early,

    CASE
        WHEN julianday(order_delivered_customer_date)
             > julianday(order_estimated_delivery_date)
        THEN 1
        ELSE 0
    END AS is_late

FROM orders
WHERE order_delivered_customer_date IS NOT NULL
""", conn)

task7_delivery.head(10)


In [ ]:
task7_state = pd.read_sql_query("""
SELECT
    c.customer_state,

    COUNT(DISTINCT o.order_id) AS delivered_orders,

    SUM(
        CASE
            WHEN julianday(o.order_delivered_customer_date)
                 > julianday(o.order_estimated_delivery_date)
            THEN 1
            ELSE 0
        END
    ) AS late_orders,

    ROUND(
        100.0 *
        SUM(
            CASE
                WHEN julianday(o.order_delivered_customer_date)
                     > julianday(o.order_estimated_delivery_date)
                THEN 1
                ELSE 0
            END
        )
        / COUNT(DISTINCT o.order_id),
        2
    ) AS late_delivery_rate

FROM orders o
JOIN customers c
    ON o.customer_id = c.customer_id

WHERE o.order_delivered_customer_date IS NOT NULL

GROUP BY c.customer_state

ORDER BY late_delivery_rate DESC
""", conn)

task7_state


In [ ]:
task7_category = pd.read_sql_query("""
SELECT
    p.product_category_name,

    COUNT(DISTINCT o.order_id) AS delivered_orders,

    SUM(
        CASE
            WHEN julianday(o.order_delivered_customer_date)
                 > julianday(o.order_estimated_delivery_date)
            THEN 1
            ELSE 0
        END
    ) AS late_orders,

    ROUND(
        100.0 *
        SUM(
            CASE
                WHEN julianday(o.order_delivered_customer_date)
                     > julianday(o.order_estimated_delivery_date)
                THEN 1
                ELSE 0
            END
        )
        / COUNT(DISTINCT o.order_id),
        2
    ) AS late_delivery_rate

FROM orders o

JOIN order_items oi
    ON o.order_id = oi.order_id

JOIN products p
    ON oi.product_id = p.product_id

WHERE o.order_delivered_customer_date IS NOT NULL
  AND p.product_category_name IS NOT NULL

GROUP BY p.product_category_name

HAVING COUNT(DISTINCT o.order_id) >= 50

ORDER BY late_delivery_rate DESC
""", conn)

task7_category


# Task 8 — Customer Experience Link — Owner: روان إبراهيم


In [ ]:
query_task8_delivery_reviews = """
WITH review_per_order AS (
    SELECT
        order_id,
        AVG(review_score) AS avg_review_score
    FROM order_reviews
    GROUP BY order_id
)

SELECT
    o.order_id,
    o.order_purchase_timestamp,
    o.order_delivered_customer_date,
    o.order_estimated_delivery_date,
    r.avg_review_score,

    CASE
        WHEN date(o.order_delivered_customer_date)
             < date(o.order_estimated_delivery_date)
            THEN 'Early'

        WHEN date(o.order_delivered_customer_date)
             = date(o.order_estimated_delivery_date)
            THEN 'On Time'

        WHEN date(o.order_delivered_customer_date)
             > date(o.order_estimated_delivery_date)
            THEN 'Late'
    END AS delivery_status

FROM orders AS o

INNER JOIN review_per_order AS r
    ON o.order_id = r.order_id

WHERE o.order_status = 'delivered'
  AND o.order_delivered_customer_date IS NOT NULL
  AND o.order_estimated_delivery_date IS NOT NULL;
"""

task8_delivery = pd.read_sql_query(
    query_task8_delivery_reviews,
    conn
)

display(task8_delivery.head(10))

In [ ]:
query_task8_delivery_summary = """
WITH review_per_order AS (
    SELECT
        order_id,
        AVG(review_score) AS avg_review_score
    FROM order_reviews
    GROUP BY order_id
),

delivery_data AS (
    SELECT
        o.order_id,
        r.avg_review_score,

        CASE
            WHEN date(o.order_delivered_customer_date)
                 < date(o.order_estimated_delivery_date)
                THEN 'Early'

            WHEN date(o.order_delivered_customer_date)
                 = date(o.order_estimated_delivery_date)
                THEN 'On Time'

            ELSE 'Late'
        END AS delivery_status

    FROM orders AS o

    INNER JOIN review_per_order AS r
        ON o.order_id = r.order_id

    WHERE o.order_status = 'delivered'
      AND o.order_delivered_customer_date IS NOT NULL
      AND o.order_estimated_delivery_date IS NOT NULL
)

SELECT
    delivery_status,
    COUNT(*) AS total_orders,
    ROUND(AVG(avg_review_score), 2) AS avg_review_score

FROM delivery_data

GROUP BY delivery_status

ORDER BY avg_review_score DESC;
"""

delivery_summary = pd.read_sql_query(
    query_task8_delivery_summary,
    conn
)

display(delivery_summary)

In [ ]:
  query_task8_poor_reviews = """
WITH review_per_order AS (
    SELECT
        order_id,
        AVG(review_score) AS avg_review_score
    FROM order_reviews
    GROUP BY order_id
),

delivery_data AS (
    SELECT
        o.order_id,
        r.avg_review_score,

        CASE
            WHEN date(o.order_delivered_customer_date)
                 < date(o.order_estimated_delivery_date)
                THEN 'Early'

            WHEN date(o.order_delivered_customer_date)
                 = date(o.order_estimated_delivery_date)
                THEN 'On Time'

            ELSE 'Late'
        END AS delivery_status

    FROM orders AS o

    INNER JOIN review_per_order AS r
        ON o.order_id = r.order_id

    WHERE o.order_status = 'delivered'
      AND o.order_delivered_customer_date IS NOT NULL
      AND o.order_estimated_delivery_date IS NOT NULL
)

SELECT
    delivery_status,
    COUNT(*) AS total_orders,

    SUM(
        CASE
            WHEN avg_review_score <= 2 THEN 1
            ELSE 0
        END
    ) AS poor_review_orders,

    ROUND(
        100.0 *
        SUM(
            CASE
                WHEN avg_review_score <= 2 THEN 1
                ELSE 0
            END
        )
        / COUNT(*),
        2
    ) AS poor_review_rate_pct

FROM delivery_data

GROUP BY delivery_status

ORDER BY poor_review_rate_pct DESC;
"""

poor_review_summary = pd.read_sql_query(
    query_task8_poor_reviews,
    conn
)

display(poor_review_summary)

In [ ]:
query_task8_freight = """
WITH item_per_order AS (
    SELECT
        order_id,
        SUM(price) AS item_value,
        SUM(freight_value) AS freight_value
    FROM order_items
    GROUP BY order_id
),

review_per_order AS (
    SELECT
        order_id,
        AVG(review_score) AS avg_review_score
    FROM order_reviews
    GROUP BY order_id
)

SELECT
    i.order_id,
    i.item_value,
    i.freight_value,

    ROUND(
        100.0 * i.freight_value /
        NULLIF(i.item_value, 0),
        2
    ) AS freight_ratio_pct,

    r.avg_review_score

FROM item_per_order AS i

INNER JOIN review_per_order AS r
    ON i.order_id = r.order_id;
"""

task8_freight = pd.read_sql_query(
    query_task8_freight,
    conn
)

display(task8_freight.head(10))

In [ ]:
query_task8_freight_bands = """
WITH item_per_order AS (
    SELECT
        order_id,
        SUM(price) AS item_value,
        SUM(freight_value) AS freight_value
    FROM order_items
    GROUP BY order_id
),

review_per_order AS (
    SELECT
        order_id,
        AVG(review_score) AS avg_review_score
    FROM order_reviews
    GROUP BY order_id
),

freight_data AS (
    SELECT
        i.order_id,

        100.0 * i.freight_value /
        NULLIF(i.item_value, 0) AS freight_ratio,

        r.avg_review_score

    FROM item_per_order AS i

    INNER JOIN review_per_order AS r
        ON i.order_id = r.order_id
)

SELECT

    CASE
        WHEN freight_ratio < 10 THEN 'Low (<10%)'
        WHEN freight_ratio < 30 THEN 'Medium (10-30%)'
        ELSE 'High (30%+)'
    END AS freight_band,

    COUNT(*) AS total_orders,

    ROUND(
        AVG(avg_review_score),
        2
    ) AS avg_review_score,

    ROUND(
        100.0 *
        SUM(
            CASE
                WHEN avg_review_score <= 2 THEN 1
                ELSE 0
            END
        )
        / COUNT(*),
        2
    ) AS poor_review_rate_pct

FROM freight_data

GROUP BY freight_band

ORDER BY avg_review_score DESC;
"""

freight_summary = pd.read_sql_query(
    query_task8_freight_bands,
    conn
)

display(freight_summary)

In [ ]:
query_task8_categories = """
WITH review_per_order AS (
    SELECT
        order_id,
        AVG(review_score) AS avg_review_score
    FROM order_reviews
    GROUP BY order_id
),

order_categories AS (
    SELECT DISTINCT
        oi.order_id,
        p.product_category_name
    FROM order_items AS oi

    INNER JOIN products AS p
        ON oi.product_id = p.product_id

    WHERE p.product_category_name IS NOT NULL
)

SELECT
    oc.product_category_name,
    COUNT(DISTINCT oc.order_id) AS total_orders,

    ROUND(
        AVG(r.avg_review_score),
        2
    ) AS avg_review_score,

    ROUND(
        100.0 *
        SUM(
            CASE
                WHEN r.avg_review_score <= 2 THEN 1
                ELSE 0
            END
        )
        / COUNT(*),
        2
    ) AS poor_review_rate_pct

FROM order_categories AS oc

INNER JOIN review_per_order AS r
    ON oc.order_id = r.order_id

GROUP BY
    oc.product_category_name

HAVING COUNT(DISTINCT oc.order_id) >= 100

ORDER BY
    poor_review_rate_pct DESC

LIMIT 15;
"""

category_summary = pd.read_sql_query(
    query_task8_categories,
    conn
)

display(category_summary)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 5))

plt.bar(
    delivery_summary["delivery_status"],
    delivery_summary["avg_review_score"]
)

plt.title("Average Review Score by Delivery Status")
plt.xlabel("Delivery Status")
plt.ylabel("Average Review Score")
plt.tight_layout()

plt.show()

In [ ]:
plt.figure(figsize=(8, 5))

plt.bar(
    poor_review_summary["delivery_status"],
    poor_review_summary["poor_review_rate_pct"]
)

plt.title("Poor Review Rate by Delivery Status")
plt.xlabel("Delivery Status")
plt.ylabel("Poor Review Rate (%)")
plt.tight_layout()

plt.show()

In [ ]:
plt.figure(figsize=(8, 5))

plt.bar(
    freight_summary["freight_band"],
    freight_summary["avg_review_score"]
)

plt.title("Average Review Score by Freight Ratio")
plt.xlabel("Freight Ratio Band")
plt.ylabel("Average Review Score")
plt.tight_layout()

plt.show()

# Task 9 — Repeat-Customer Analysis — Owner: إبراهيم


In [ ]:
# Task 9: تحليل العملاء المتكررين والجدد
query_task_9 = """
WITH customer_order_counts AS (
    SELECT 
        c.customer_unique_id,
        COUNT(DISTINCT o.order_id) AS order_count
    FROM orders o
    JOIN customers c ON o.customer_id = c.customer_id
    GROUP BY c.customer_unique_id
),
customer_segments AS (
    SELECT 
        customer_unique_id,
        order_count,
        CASE 
            WHEN order_count = 1 THEN 'One-time Customer'
            ELSE 'Repeat Customer'
        END AS customer_type
    FROM customer_order_counts
)
SELECT 
    customer_type,
    COUNT(customer_unique_id) AS customer_count,
    ROUND(COUNT(customer_unique_id) * 100.0 / (SELECT COUNT(*) FROM customer_segments), 2) AS percentage_share
FROM customer_segments
GROUP BY customer_type;
"""

df_repeat_customer = pd.read_sql_query(query_task_9, conn)
df_repeat_customer


# Task 10 — Freight Efficiency — Owner: لقاء


1. Freight Efficiency by State

In [ ]:
query = """
SELECT
    c.customer_state,
    SUM(oi.price) AS total_item_value,
    SUM(oi.freight_value) AS total_freight,
    SUM(oi.freight_value) * 100.0 / SUM(oi.price) AS freight_ratio
FROM order_items oi
JOIN orders o
    ON oi.order_id = o.order_id
JOIN customers c
    ON o.customer_id = c.customer_id
WHERE oi.price > 0
GROUP BY c.customer_state
ORDER BY freight_ratio DESC;
"""

state_df = pd.read_sql_query(query, conn)
state_df


 📊 Freight Efficiency by State

In [ ]:
import matplotlib.pyplot as plt

plt.barh(state_df["customer_state"], state_df["freight_ratio"])

plt.xlabel("Freight Ratio (%)")
plt.ylabel("Customer State")
plt.title("Freight Efficiency by State")

plt.show()


3. Freight Efficiency by Category

In [ ]:
query = """
SELECT
    p.product_category_name,
    SUM(oi.price) AS total_item_value,
    SUM(oi.freight_value) AS total_freight,
    SUM(oi.freight_value) * 100.0 / SUM(oi.price) AS freight_ratio
FROM order_items oi
JOIN products p
    ON oi.product_id = p.product_id
WHERE oi.price > 0
GROUP BY p.product_category_name
ORDER BY freight_ratio DESC;
"""

pd.read_sql_query(query, conn)


4. Freight Efficiency by Seller

In [ ]:
query = """
SELECT
    seller_id,
    COUNT(DISTINCT order_id) AS total_orders,
    SUM(price) AS total_item_value,
    SUM(freight_value) AS total_freight,
    SUM(freight_value) * 100.0 / SUM(price) AS freight_ratio
FROM order_items
WHERE price > 0
GROUP BY seller_id
HAVING COUNT(DISTINCT order_id) >= 10
ORDER BY freight_ratio DESC;
"""

pd.read_sql_query(query, conn)


# Task 11 — Window-Function Ranking — Owner: روان موسى


In [ ]:
task11 = pd.read_sql_query("""
WITH seller_state_sales AS (

    SELECT
        c.customer_state,
        oi.seller_id,

        SUM(oi.price) AS revenue,

        COUNT(DISTINCT oi.order_id) AS orders

    FROM order_items oi

    JOIN orders o
        ON oi.order_id = o.order_id

    JOIN customers c
        ON o.customer_id = c.customer_id

    GROUP BY
        c.customer_state,
        oi.seller_id

    HAVING COUNT(DISTINCT oi.order_id) >= 50
),

ranked_sellers AS (

    SELECT
        customer_state,
        seller_id,
        revenue,
        orders,

        RANK() OVER (
            PARTITION BY customer_state
            ORDER BY revenue DESC
        ) AS seller_rank

    FROM seller_state_sales
)

SELECT *
FROM ranked_sellers
WHERE seller_rank <= 3

ORDER BY customer_state, seller_rank
""", conn)

task11


# Task 12 — Reusable Analytics View — Owner: روان إبراهيم


In [ ]:
create_view_query = """
DROP VIEW IF EXISTS vw_order_summary;

CREATE VIEW vw_order_summary AS

WITH item_per_order AS (
    SELECT
        order_id,
        COUNT(*) AS total_items,
        SUM(price) AS item_revenue,
        SUM(freight_value) AS total_freight
    FROM order_items
    GROUP BY order_id
),

payment_per_order AS (
    SELECT
        order_id,
        SUM(payment_value) AS total_payment,
        MAX(payment_installments) AS max_installments,
        COUNT(*) AS payment_records
    FROM order_payments
    GROUP BY order_id
),

review_per_order AS (
    SELECT
        order_id,
        AVG(review_score) AS avg_review_score,
        COUNT(*) AS review_count
    FROM order_reviews
    GROUP BY order_id
)

SELECT
    o.order_id,
    o.customer_id,
    o.order_status,
    o.order_purchase_timestamp,
    o.order_delivered_customer_date,
    o.order_estimated_delivery_date,

    CASE
        WHEN o.order_delivered_customer_date IS NULL
          OR o.order_estimated_delivery_date IS NULL
            THEN NULL

        WHEN date(o.order_delivered_customer_date)
             < date(o.order_estimated_delivery_date)
            THEN 'Early'

        WHEN date(o.order_delivered_customer_date)
             = date(o.order_estimated_delivery_date)
            THEN 'On Time'

        ELSE 'Late'
    END AS delivery_status,

    COALESCE(i.total_items, 0) AS total_items,
    COALESCE(i.item_revenue, 0) AS item_revenue,
    COALESCE(i.total_freight, 0) AS total_freight,

    COALESCE(p.total_payment, 0) AS total_payment,
    p.max_installments,
    COALESCE(p.payment_records, 0) AS payment_records,

    r.avg_review_score,
    COALESCE(r.review_count, 0) AS review_count

FROM orders AS o

LEFT JOIN item_per_order AS i
    ON o.order_id = i.order_id

LEFT JOIN payment_per_order AS p
    ON o.order_id = p.order_id

LEFT JOIN review_per_order AS r
    ON o.order_id = r.order_id;
"""

conn.executescript(create_view_query)

print("vw_order_summary created successfully")

In [ ]:
view_preview = pd.read_sql_query(
    """
    SELECT *
    FROM vw_order_summary
    LIMIT 10;
    """,
    conn
)

display(view_preview)

In [ ]:
view_validation = pd.read_sql_query(
    """
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT order_id) AS distinct_orders,
        COUNT(*) - COUNT(DISTINCT order_id) AS duplicate_orders
    FROM vw_order_summary;
    """,
    conn
)

display(view_validation)

In [ ]:
count_check = pd.read_sql_query(
    """
    SELECT
        (SELECT COUNT(*) FROM orders) AS orders_table_count,
        (SELECT COUNT(*) FROM vw_order_summary) AS view_count;
    """,
    conn
)

display(count_check)

In [ ]:
summary_check = pd.read_sql_query(
    """
    SELECT
        delivery_status,
        COUNT(*) AS total_orders,
        ROUND(AVG(total_payment), 2) AS avg_payment,
        ROUND(AVG(avg_review_score), 2) AS avg_review_score
    FROM vw_order_summary
    WHERE delivery_status IS NOT NULL
    GROUP BY delivery_status;
    """,
    conn
)

display(summary_check)